In [1]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
import torch
import torch.nn as nn
import torch.nn.functional as F

c:\Users\ps302\anaconda3\envs\genai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
device

device(type='cpu')

In [3]:
text_encoder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

text_encoder.to(device)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3528.80it/s]


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)

In [4]:
print(text_encoder.get_embedding_dimension())

384


In [10]:
# text_embeddings[0]
text_01 = "Applying deep learning to satellite imagery for advanced flood forecasting. This research integrates neural networks with remote sensing data to model complex hydrological systems"
text_02="Utilizing machine learning models on orbital data allows for enhanced prediction of flooding events. This approach merges artificial intelligence with satellite observations to capture intricate patterns in river basins."
text_03="Manually surveying localized drought conditions on the ground eliminates the need for computerized modeling. This strategy completely ignores automated data structures and relies solely on historical paper records."

In [11]:
text_embeddind_01 = text_encoder.encode(
    text_01,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)

text_embeddind_01

Batches: 100%|██████████| 1/1 [00:00<00:00, 100.06it/s]


tensor([-7.0280e-02, -1.3017e-01,  1.3838e-01,  5.6039e-03,  1.0237e-02,
        -1.8181e-02, -7.6327e-02, -2.6740e-02,  2.6647e-02, -1.4730e-03,
        -1.1382e-01, -4.5277e-02, -9.2053e-03,  4.7166e-02, -6.2040e-02,
         2.5824e-02, -1.2398e-01, -2.6323e-02, -5.5537e-02, -5.3855e-02,
         4.6715e-02,  6.7601e-02, -1.2003e-01, -3.9712e-02,  4.5786e-02,
         6.5659e-02,  1.8574e-02, -8.1966e-03, -1.1116e-02,  4.1776e-02,
         4.2114e-02,  6.3963e-03, -7.3960e-02,  2.8127e-02, -4.5198e-03,
         1.4778e-01, -4.7231e-02, -2.1401e-03, -1.0352e-02,  2.0924e-02,
         7.4313e-02, -5.3877e-02,  5.2146e-02, -1.1900e-02,  5.2073e-02,
         5.5664e-02,  1.6890e-02, -6.7524e-02,  6.2863e-02,  1.4103e-02,
        -2.2686e-02, -1.9131e-02, -2.1411e-02,  4.4115e-02, -6.8795e-03,
        -7.3780e-03, -2.6384e-04,  6.8643e-03, -4.6911e-02,  2.3409e-03,
        -7.2451e-04,  1.0328e-02, -5.3100e-03,  2.3527e-02,  2.5778e-02,
         7.4207e-02, -8.9661e-02,  8.8390e-02,  3.6

In [12]:
text_embeddind_02 = text_encoder.encode(
    text_02,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)

text_embeddind_02

Batches: 100%|██████████| 1/1 [00:00<00:00, 42.51it/s]


tensor([-3.5749e-02, -6.2969e-02,  1.2300e-01, -3.2151e-04, -1.6689e-02,
        -5.4635e-02, -3.2410e-02, -4.1405e-02, -4.1818e-03,  6.9366e-02,
        -1.2474e-01, -5.1521e-02, -1.5790e-02,  3.0227e-02, -4.8783e-02,
         6.4948e-02, -5.7127e-02, -2.1917e-02, -3.9378e-03, -9.1460e-02,
         4.6069e-02,  3.1276e-02, -1.3920e-01, -1.2459e-02,  5.7615e-02,
         8.9559e-02,  6.1248e-02,  2.8510e-02, -3.8352e-02,  4.3141e-02,
         4.4784e-02, -1.9562e-02, -4.6745e-02, -4.8819e-02, -2.8503e-02,
         7.0199e-02, -9.9576e-02,  2.7291e-03,  4.0916e-03,  4.6615e-03,
         8.7455e-02, -3.2371e-02,  9.3604e-02, -6.3497e-03,  4.7182e-02,
         5.6911e-02, -2.0625e-02, -7.5051e-02,  3.0339e-04,  3.4192e-02,
        -1.8731e-02, -4.3556e-02,  1.7249e-02,  1.3233e-04, -6.5245e-02,
        -4.8480e-02, -4.3268e-02, -4.7732e-02,  4.8455e-02, -6.9607e-02,
        -5.0636e-03, -2.9766e-02, -3.0158e-02,  3.6304e-02,  2.0108e-03,
         5.8569e-02, -7.7891e-02,  1.1021e-01,  4.8

In [13]:
text_embeddind_03 = text_encoder.encode(
    text_03,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)

text_embeddind_03

Batches: 100%|██████████| 1/1 [00:00<00:00, 38.43it/s]


tensor([-2.3738e-03,  6.8510e-03,  7.8501e-02,  6.4595e-02,  1.3199e-02,
        -9.3957e-02, -9.8473e-02, -9.3264e-03,  7.2520e-04,  2.2384e-02,
        -4.2310e-02, -4.2010e-02, -2.0836e-03,  5.1777e-02, -6.3009e-02,
        -1.8945e-02, -1.5351e-01, -3.4050e-02, -3.9121e-03, -8.1492e-02,
         2.0842e-02,  2.1542e-02, -9.5451e-02, -1.7617e-02,  5.7471e-02,
         3.4473e-02, -2.6810e-02,  3.1263e-03, -4.4577e-02,  5.6261e-02,
        -8.1969e-02,  4.7184e-02,  9.6819e-02,  1.6833e-02,  3.1258e-02,
        -4.2376e-02,  6.7604e-03,  1.9308e-03, -1.1553e-01,  1.8355e-02,
        -2.4065e-02, -4.2584e-03,  6.1811e-02, -1.0866e-02,  4.9833e-02,
         1.5006e-02, -3.3866e-02, -2.2357e-02, -5.7239e-02,  5.9038e-02,
         8.5408e-03,  3.5454e-02, -1.8432e-02, -3.2690e-02,  1.5269e-02,
        -2.8946e-02,  7.2680e-02, -9.8934e-02,  1.0369e-02,  2.3157e-02,
         4.3179e-02,  3.0116e-02, -9.7245e-02,  4.7872e-03,  6.2580e-02,
         7.0028e-02, -6.8801e-02,  4.1799e-02,  6.9

In [14]:
similarity_score_1 = torch.dot(text_embeddind_01, text_embeddind_02)
similarity_score_2 = torch.dot(text_embeddind_01, text_embeddind_03)
print(similarity_score_1, similarity_score_2)

tensor(0.6826) tensor(0.3252)
